<h3>Setup: imports and molecule path</h3>

In [ ]:
# Setup: imports and molecule path
from pathlib import Path
import numpy as np

import qdk_chemistry.plugins.pyscf  # Required for scf_solver

from qdk_chemistry.data import Structure
from qdk_chemistry.algorithms import available, create, print_settings
from qdk_chemistry.constants import ANGSTROM_TO_BOHR
from qdk_chemistry.utils import Logger

Logger.set_global_level(Logger.LogLevel.off)

N2_XYZ = Path("../examples/data/stretched_n2.structure.xyz")

<h3>Load N₂ structure from file and from coordinates</h3>

In [ ]:
# Load N₂ structure from file and from coordinates
structure = Structure.from_xyz_file(N2_XYZ)
print("Loaded from file:")
print(structure.to_xyz())

# Method 2: build directly from coordinates (in Bohr)
# 1.27 Å × ANGSTROM_TO_BOHR (≈ 1.889 Bohr/Å) = 2.4008 Bohr
N2_BOND_BOHR = 1.27 * ANGSTROM_TO_BOHR
structure_manual = Structure(
    coordinates=np.array([[0.0, 0.0, 0.0], [0.0, 0.0, N2_BOND_BOHR]]),
    symbols=["N", "N"]
)
print("\nBuilt from coordinates (Bohr input):")
print(structure_manual.to_xyz())

<h3>Run SCF with STO-3G basis</h3>

In [ ]:
# Run SCF with STO-3G basis
scf_solver = create("scf_solver")
E_hf_sto3g, wfn_sto3g = scf_solver.run(
    structure,
    charge=0,
    spin_multiplicity=1,
    basis_or_guess="sto-3g"
)

print(f"HF energy (sto-3g): {E_hf_sto3g:.6f} Hartree")

<h3>Explore the Wavefunction and Orbitals objects</h3>

In [ ]:
# Explore the Wavefunction and Orbitals objects
num_alpha, num_beta = wfn_sto3g.get_total_num_electrons()
print(f"Total electrons: alpha={num_alpha}, beta={num_beta}")

# Orbital summary
orbitals = wfn_sto3g.get_orbitals()
print("\nOrbital summary (sto-3g):")
print(orbitals.get_summary())

<h3>Run SCF with cc-pVDZ and compare energies</h3>

In [ ]:
# Run SCF with cc-pVDZ and compare energies
scf_solver_dz = create("scf_solver")
E_hf_dz, wfn_dz = scf_solver_dz.run(
    structure,
    charge=0,
    spin_multiplicity=1,
    basis_or_guess="cc-pvdz"
)

print(f"HF energy (sto-3g):  {E_hf_sto3g:.6f} Hartree")
print(f"HF energy (cc-pvdz): {E_hf_dz:.6f} Hartree")
print(f"Basis set effect:    {E_hf_dz - E_hf_sto3g:.6f} Hartree")

<h3>Inspect cc-pVDZ orbital summary</h3>

In [ ]:
# Inspect cc-pVDZ orbital summary
orbitals_dz = wfn_dz.get_orbitals()
print("Orbital summary (cc-pvdz):")
print(orbitals_dz.get_summary())

# YOUR CODE: How many more orbitals does cc-pvdz produce vs sto-3g?
# print(f"Orbital count increase: {???} orbitals")

<h3>Run DFT and compare with Hartree-Fock</h3>

In [ ]:
# Run DFT and compare with Hartree-Fock
scf_dft = create("scf_solver", "pyscf", method="???") 
E_dft, wfn_dft = scf_dft.run(structure, charge=0, spin_multiplicity=1, basis_or_guess="cc-pvdz")

print(f"DFT energy (cc-pvdz): {E_dft:.6f} Hartree")
print(f"HF  energy (cc-pvdz): {E_hf_dz:.6f} Hartree")
print(f"Difference:            {E_dft - E_hf_dz:.6f} Hartree")

<h3>Export QDK QubitHamiltonian to Qiskit</h3>

In [ ]:
# Export QDK QubitHamiltonian to Qiskit
import qdk_chemistry.plugins.qiskit
from qdk_chemistry.utils import compute_valence_space_parameters

# Build a minimal active space from the STO-3G wavefunction (already computed in Part 2)
num_val_e, num_val_o = compute_valence_space_parameters(wfn_sto3g, charge=0)
wfn_valence = create("active_space_selector", "qdk_valence",
                     num_active_electrons=num_val_e,
                     num_active_orbitals=num_val_o).run(wfn_sto3g)
hamiltonian = create("hamiltonian_constructor").run(wfn_valence.get_orbitals())

# Map to qubits using the Qiskit Jordan-Wigner mapper
qubit_mapper = create("qubit_mapper", "qiskit", encoding="jordan-wigner")
qubit_hamiltonian = qubit_mapper.run(hamiltonian)

print(f"QDK QubitHamiltonian: {len(qubit_hamiltonian.pauli_strings)} Pauli strings")

# pauli_ops is a native Qiskit SparsePauliOp—usable directly in Qiskit VQE, Estimator, etc.
qiskit_op = qubit_hamiltonian.pauli_ops
print(f"Qiskit object type:   {type(qiskit_op).__name__}")
print(f"Num qubits:           {qiskit_op.num_qubits}")
print(f"\nFirst 3 Pauli terms:")
for pauli, coeff in list(zip(qiskit_op.paulis, qiskit_op.coeffs))[:3]:
    print(f"  {pauli}  coeff={coeff:.6f}")

<h3>Import Qiskit SparsePauliOp into the QDK</h3>

In [ ]:
# Import Qiskit SparsePauliOp into the QDK
import numpy as np                                                                                                                        
from qiskit.quantum_info import SparsePauliOp                                                                                             
from qiskit import QuantumCircuit, qasm3                                                                                                  
from qiskit.circuit.library import StatePreparation as QiskitStatePreparation                                                             
from qdk_chemistry.data import Circuit, QubitHamiltonian
from qdk_chemistry.algorithms import create

# H2 minimal basis 2-qubit Hamiltonian defined in Qiskit
h2_op = SparsePauliOp.from_list([
    ("II", -1.05342108),
    ("IZ",  0.39484436),
    ("XX",  0.18121046),
    ("ZI", -0.39484436),
    ("ZZ", -0.01124616),
])

# Bridge into QDK — fill in the blanks
qdk_h2 = QubitHamiltonian(
    pauli_strings="???",  
    coefficients="???",   
)

# Trial state: HF reference (|01⟩ in parity basis)
trial_vec = np.array([0.0, 1.0, 0.0, 0.0], dtype=complex)
qc = QuantumCircuit(2, name="trial")
qc.append(QiskitStatePreparation(trial_vec), [0, 1])
trial_circuit = Circuit(qasm3.dumps(qc))

# Run iterative QPE
iqpe = create("phase_estimation", "iterative",
              num_bits=6, evolution_time=np.pi/4, shots_per_bit=3)
simulator = create("circuit_executor", "qiskit_aer_simulator", seed=42)
evolution_builder = create("time_evolution_builder", "trotter")
circuit_mapper = create("controlled_evolution_circuit_mapper", "pauli_sequence")

result = iqpe.run(
    state_preparation=trial_circuit,
    qubit_hamiltonian=qdk_h2,
    circuit_executor=simulator,
    evolution_builder=evolution_builder,
    circuit_mapper=circuit_mapper,
)

energy = result.resolved_energy if result.resolved_energy is not None else result.raw_energy
NUCLEAR_REPULSION = 0.71510434 # hardcoded for now for H2 at 0.74 Å
print(f"Total ground state energy: {energy + NUCLEAR_REPULSION:.4f} Ha")